In [1]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0918\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[ERROR] Google API 키 미설정
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [15]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [6]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [7]:
chat_message_history = SQLChatMessageHistory(
    session_id = "sql_history",
    connection = "sqlite:///sqlite.db"
)

In [11]:
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. "
    "내 이름은 테디야. "
    "나는 랭체인 개발자야. "
    "앞으로 잘 부탁해!"
)

chat_message_history.add_ai_message(
    "안녕 테디, 만나서 반가워. 나도 잘 부탁해!"
)

print(chat_message_history.messages)

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
    
        MessagesPlaceholder(variable_name="chat_history"),
        
        ("human", "{question}"),
    ]
)

In [16]:
chain = prompt | llm | StrOutputParser()

In [17]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name = user_id,
        session_id = conversation_id,
        connection = "sqlite:///sqlite.db"
    )

In [18]:
from langchain_core.runnables import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id = "user_id",
        annotation = str,
        name = "User ID",
        description = "Unique identifier for a user.",
        default = "",
        is_shared = True,
    ),
    
    ConfigurableFieldSpec(
        id = "conversation_id",
        annotation = str,
        name = "Conversation ID",
        description = "Unique identifier for a conversation.",
        default = "",
        is_shared = True,
    ),
]

In [21]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key = "question",
    history_messages_key = "chat_history",
    history_factory_config = config_fields,
)

In [22]:
config = {
    "configurable": {
        "user_id": "user1",
        "conversation_id": "conversation1",
    }
}

In [23]:
response = chain_with_history.invoke(
    {
        "question": "안녕 반가워, 내 이름은 테디야"
    },
    config=config,
)

print(response)

안녕 테디야! 반가워. 난 AI 어시스턴트야 — 앞으로 대화할 때는 테디라고 부를게. 뭐 도와줄까?


In [24]:
config = {
    "configurable": {
        "user_id": "user1",
        "conversation_id": "conversation2",
    }
}

In [25]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고?"},
    config=config
)

'저는 지금 당신의 이름을 알지 못해요. 저는 사용자의 개인 정보를 자동으로 알 수 없으니, 원하시면 이름을 알려 주세요 — 그러면 이 채팅에서 그렇게 불러드릴게요.\n\n어떻게 불러드릴까요? (예: 성함, 이름+님, 애칭 등)'